Instalaciones: 

In [1]:
import pandas as pd
import numpy as np


Carga de archivos de validación y de probabilidades

In [2]:
pd_validacion = pd.read_csv("validacion.csv")
y_val = np.array(pd_validacion["klass"])


In [3]:
pd_Gemma = pd.read_csv("LLM_probs_umbral_0.89_app.csv")
probs_Gemma = np.array(pd_Gemma['klass'])
pd_Gemma_1 = pd.read_csv("LLM_8791_validacion.csv")
probs_Gemma_1 = np.array(pd_Gemma_1['klass'])
pd_betito = pd.read_csv("robertuito_probs_3.1e-05_565_158_0.001278_32_0.153572_0.136504_val.csv")
probs_betito = np.array(pd_betito['klass'])
pd_Osth = pd.read_csv("Qwen_OSTH_finetuned_scores_val.csv")
probs_Osth = np.array(pd_Osth["score_qwen"])
pd_Llama = pd.read_csv("Llama_AntiHumor_scores_val.csv")
probs_Llama = np.array(pd_Llama["score_antih"])


In [4]:
pd_Gemma_test = pd.read_csv("LLM_probs_0.54_app_test.csv")
test_Gemma = np.array(pd_Gemma_test['klass'])
pd_Gemma_1_test = pd.read_csv("LLM_8791_test.csv")
test_Gemma_1 = np.array(pd_Gemma_1_test['klass'])
pd_betito_test = pd.read_csv("robertuito_probs_3.1e-05_565_158_0.001278_32_0.153572_0.136504_test.csv")
test_betito = np.array(pd_betito_test['klass'])
pd_Osth_test = pd.read_csv("Qwen_OSTH_finetuned_scores_test_V2.csv")
test_Osth = np.array(pd_Osth_test["score_qwen"])
pd_Llama_test = pd.read_csv("Llama_AntiHumor_scores_test.csv")
test_Llama = np.array(pd_Llama_test["score_antih"])

Busqueda en dos modelos

In [ ]:
import numpy as np
from sklearn.metrics import f1_score

p_gemma_1   = probs_Gemma_1
p_roberta = probs_betito

# Búsqueda de pesos óptimos en validación
best_w, best_f1 = 0.5, 0.0
for w in np.arange(0.1, 1.0, 0.05):
    p_ens = w * p_gemma_1 + (1 - w) * p_roberta
    # también buscamos umbral óptimo para cada peso
    for thresh in np.arange(0.3, 0.7, 0.02):
        preds = (p_ens >= thresh).astype(int)
        f1 = f1_score(y_val, preds, average='macro')
        if f1 > best_f1:
            best_f1, best_w, best_thresh = f1, w, thresh

print(f"Mejor peso Gemma: {best_w:.2f}  Umbral: {best_thresh:.2f}  F1 val: {best_f1:.4f}")

Mejor peso Gemma: 0.70  Umbral: 0.68  F1 val: 0.8804


In [ ]:
import numpy as np
import pandas as pd

best_w      = 0.70
best_thresh = 0.68

p_gemma_test = test_Gemma
p_gemma_test_1 = test_Gemma_1
p_roberta_test = test_betito

# Ensemble sobre test
p_ensemble_test = best_w * p_gemma_test_1 + (1 - best_w) * p_roberta_test

# Predicciones finales
test_preds_ensemble = (p_ensemble_test >= best_thresh).astype(int)

unique, counts = np.unique(test_preds_ensemble, return_counts=True)
print(dict(zip(unique.tolist(), counts.tolist())))

# Guardar CSV

pd.DataFrame({
    'id'    : range(1, len(test_preds_ensemble) + 1),
    'klass' : test_preds_ensemble,
    #'score' : p_ensemble_test,
}).to_csv('GG_Robertuito_V5.csv', index=False)

print('GG_Robertuito_V5.csv listo')

Busqueda en tres modelos

In [ ]:
import numpy as np
from sklearn.metrics import f1_score


p_gemma_old = probs_betito    
p_gemma_new = probs_Gemma_1  
p_roberta   = probs_Llama  

best_weights = (0.33, 0.33, 0.34)
best_thresh = 0.5
best_f1 = 0.0

print("Buscando la combinación óptima de pesos y umbral...")

# Grid Search
for w_gemma_old in np.arange(0.0, 1.01, 0.05):
    for w_gemma_new in np.arange(0.0, 1.01 - w_gemma_old, 0.05):
        
        w_roberta = 1.0 - w_gemma_old - w_gemma_new

        if w_roberta < 0.0:
            continue

        p_ens = (w_gemma_old * p_gemma_old) + (w_gemma_new * p_gemma_new) + (w_roberta * p_roberta)

        # Búsqueda del umbral óptimo de clasificación
        for thresh in np.arange(0.3, 0.71, 0.02):
            preds = (p_ens >= thresh).astype(int)

            f1 = f1_score(y_val, preds, average='macro')

            # Si encontramos un F1-Score más alto, guardamos la configuración
            if f1 > best_f1:
                best_f1 = f1
                best_weights = (w_gemma_old, w_gemma_new, w_roberta)
                best_thresh = thresh

w_old, w_new, w_rob = best_weights
print("\n¡Búsqueda completada con éxito!")
print("─" * 50)
print(f"Pesos óptimos encontrados:")
print(f"  • Gemma Anterior (p_gemma):   {w_old:.2f}")
print(f"  • Gemma Nuevo (p_gemma_1):   {w_new:.2f}")
print(f"  • RoBERTuito (p_roberta):     {w_rob:.2f}")
print(f"  (Suma total de pesos: {w_old + w_new + w_rob:.2f})")
print("─" * 50)
print(f"Umbral de decisión (Threshold): {best_thresh:.2f}")
print(f"Mejor Macro F1-Score en Val:    {best_f1:.4f}")
print("─" * 50)

In [ ]:
import numpy as np
import pandas as pd

p_gemma_test = test_Osth
p_gemma_test_1 = test_Gemma_1
p_roberta_test = test_betito

# Ensemble sobre test
p_ensemble_test = (best_weights[0] * p_gemma_test) + (best_weights[1] * p_gemma_test_1) + (best_weights[2] * p_roberta_test)

# Predicciones finales
test_preds_ensemble = (p_ensemble_test >= best_thresh).astype(int)

unique, counts = np.unique(test_preds_ensemble, return_counts=True)
print(dict(zip(unique.tolist(), counts.tolist())))

# Guardar CSV

NAME = 'WW_GG_Robertuito_V1.csv'

pd.DataFrame({
    'id'    : range(1, len(test_preds_ensemble) + 1),
    'klass' : test_preds_ensemble,
    #'score' : p_ensemble_test,
}).to_csv(NAME, index=False)

print(f'{NAME}')

{0: 3568, 1: 2032}
✅ WW_GG_Robertuito_V1.csv


Busqueda en 4 modelos y análisis de correlación y errores
(Mejor ensamble)

In [ ]:
import numpy as np
from sklearn.metrics import f1_score


p_gemma_old = probs_Osth       
p_gemma_new = probs_Gemma_1    
p_roberta   = probs_betito     
p_Llama = probs_Llama        

best_weights = (0.25, 0.25, 0.25, 0.25)
best_thresh = 0.5
best_f1 = 0.0

print("Buscando la combinación óptima de pesos (4 modelos) y umbral...")

for w_gemma_old in np.arange(0.0, 1.01, 0.05):
    for w_gemma_new in np.arange(0.0, 1.01 - w_gemma_old, 0.05):
        
        suma_parcial_1 = w_gemma_old + w_gemma_new
        
        for w_roberta in np.arange(0.0, 1.01 - suma_parcial_1, 0.05):
            
            w_qwen_osth = 1.0 - w_gemma_old - w_gemma_new - w_roberta
            
            if w_qwen_osth < -1e-9:
                continue
            if w_qwen_osth < 0.0:
                w_qwen_osth = 0.0  

            p_ens = (
                (w_gemma_old * p_gemma_old) + 
                (w_gemma_new * p_gemma_new) + 
                (w_roberta * p_roberta) + 
                (w_qwen_osth * p_Llama)
            )

            # Búsqueda del umbral óptimo de decisión
            for thresh in np.arange(0.3, 0.71, 0.02):
                preds = (p_ens >= thresh).astype(int)

                f1 = f1_score(y_val, preds, average='macro')

                # Si encontramos un nuevo récord de F1, guardamos la alineación
                if f1 > best_f1:
                    best_f1 = f1
                    best_weights = (w_gemma_old, w_gemma_new, w_roberta, w_qwen_osth)
                    best_thresh = thresh

w_old, w_new, w_rob, w_qwen = best_weights
print("\n¡Búsqueda de 4 vías completada con éxito! 🏆")
print("─" * 55)
print(f"Pesos óptimos encontrados:")
print(f"  • Gemma Anterior (p_gemma):    {w_old:.2f}")
print(f"  • Gemma Nuevo (p_gemma_1):    {w_new:.2f}")
print(f"  • RoBERTuito (p_roberta):      {w_rob:.2f}")
print(f"  • Llama (p_qwen_osth):     {w_qwen:.2f}")
print(f"  (Suma total de pesos: {w_old + w_new + w_rob + w_qwen:.2f})")
print("─" * 55)
print(f"Umbral de decisión (Threshold):  {best_thresh:.2f}")
print(f"Mejor Macro F1-Score en Val:     {best_f1:.4f}")
print("─" * 55)

Buscando la combinación óptima de pesos (4 modelos) y umbral...

¡Búsqueda de 4 vías completada con éxito! 🏆
───────────────────────────────────────────────────────
Pesos óptimos encontrados:
  • Gemma Anterior (p_gemma):    0.05
  • Gemma Nuevo (p_gemma_1):    0.55
  • RoBERTuito (p_roberta):      0.10
  • Llama (p_qwen_osth):     0.30
  (Suma total de pesos: 1.00)
───────────────────────────────────────────────────────
Umbral de decisión (Threshold):  0.70
Mejor Macro F1-Score en Val:     0.8940
───────────────────────────────────────────────────────


In [13]:

p_gemma_test = test_Osth
p_gemma_test_1 = test_Gemma_1
p_roberta_test = test_betito
p_Llama = test_Llama


p_ens_test = (best_weights[0] * p_gemma_test) + (best_weights[1] * p_gemma_test_1) + (best_weights[2] * p_roberta_test) + (best_weights[3] * p_Llama)
preds_finales_test = (p_ens_test >= best_thresh).astype(int)

unique, counts = np.unique(preds_finales_test, return_counts=True)
print(dict(zip(unique.tolist(), counts.tolist())))

# Guardar CSV

NAME = 'WW_GG_Robertuito_LL_V1.csv'

pd.DataFrame({
    'id'    : range(1, len(preds_finales_test) + 1),
    'klass' : preds_finales_test,
    #'score' : p_ensemble_test,
}).to_csv(NAME, index=False)

print(f'✅ {NAME} listo')

{0: 3585, 1: 2015}
✅ WW_GG_Robertuito_LL_V1.csv listo


In [14]:
def best_f1(prob, y):

    best_f1 = 0
    best_thresh = 0

    for thresh in np.arange(0.1,0.91,0.01):

        preds = (prob >= thresh).astype(int)

        f1 = f1_score(
            y,
            preds,
            average='macro'
        )

        if f1 > best_f1:
            best_f1 = f1
            best_thresh = thresh

    return best_f1, best_thresh

In [15]:
print("Gemma viejo:", best_f1(probs_Gemma,y_val))
print("Gemma nuevo:", best_f1(probs_Gemma_1,y_val))
print("RoBERTuito:", best_f1(probs_betito,y_val))
print("Qwen:", best_f1(probs_Osth,y_val))
print("BERT:", best_f1(probs_Llama,y_val))

Gemma viejo: (0.8787625931061563, np.float64(0.8899999999999996))
Gemma nuevo: (0.8808257421264574, np.float64(0.8599999999999995))
RoBERTuito: (0.8656763276384865, np.float64(0.5499999999999998))
Qwen: (0.8776714513556618, np.float64(0.4099999999999998))
BERT: (0.878097381437982, np.float64(0.6599999999999997))


In [8]:
_, t_gemma = best_f1(probs_Gemma,y_val)
_, t_gemma_1 = best_f1(probs_Gemma_1,y_val)
_, t_roberta = best_f1(probs_Osth,y_val)
_, t_qwen = best_f1(probs_Osth,y_val)

In [9]:
pred_gemma = (probs_Gemma_1 >= t_gemma).astype(int)
pred_gemma_1 = (probs_Gemma_1 >= t_gemma_1).astype(int)
pred_qwen = (probs_Osth >= t_qwen).astype(int)
pred_roberta = (probs_betito >= t_roberta).astype(int)


err_gemma = (pred_gemma != y_val).astype(int)
err_gemma_1 = (pred_gemma_1 != y_val).astype(int)
err_qwen = (pred_qwen != y_val).astype(int)
err_roberta = (pred_roberta != y_val).astype(int)

In [10]:
import pandas as pd

df = pd.DataFrame({
    "Gemma": probs_Gemma_1,
    "Qwen": probs_Osth,
    "RoBERTuito": probs_betito,
    "Gemma_old": probs_Gemma
})

print("Correlaciones")
print(df.corr())

Correlaciones
               Gemma      Qwen  RoBERTuito  Gemma_old
Gemma       1.000000  0.901010    0.852566   0.999736
Qwen        0.901010  1.000000    0.883157   0.900096
RoBERTuito  0.852566  0.883157    1.000000   0.851127
Gemma_old   0.999736  0.900096    0.851127   1.000000


In [11]:
import pandas as pd

print("Errores")
errs = pd.DataFrame({
    "Gemma": err_gemma_1,
    "Qwen": err_qwen,
    "RoBERTuito": err_roberta,
    "Gemma_old": err_gemma
})

print(errs.corr())

Errores
               Gemma      Qwen  RoBERTuito  Gemma_old
Gemma       1.000000  0.644714    0.541682   0.936974
Qwen        0.644714  1.000000    0.580181   0.627192
RoBERTuito  0.541682  0.580181    1.000000   0.534790
Gemma_old   0.936974  0.627192    0.534790   1.000000


In [12]:
solo_roberta = (
    (pred_roberta == y_val) &
    (pred_gemma != y_val) &
    (pred_qwen != y_val) &
    (pred_gemma_1 != y_val)
)

print(f"Solo roberta: {solo_roberta.sum()}")
solo_qwen = (
    (pred_qwen == y_val) &
    (pred_gemma != y_val) &
    (pred_roberta != y_val) &
    (pred_gemma_1 != y_val)
)

print(f"Solo qween: {solo_qwen.sum()}")

solo_gemma = (
    (pred_gemma == y_val) &
    (pred_qwen != y_val) &
    (pred_roberta != y_val) &
    (pred_gemma_1 != y_val)
)

print(f"Solo Gemma {solo_gemma.sum()}")

solo_gemma_1 = (
    (pred_gemma_1 == y_val) &
    (pred_qwen != y_val) &
    (pred_roberta != y_val) &
    (pred_gemma != y_val)
)

print(f"Solo Gemma Nuevo {solo_gemma_1.sum()}")

Solo roberta: 17
Solo qween: 11
Solo Gemma 1
Solo Gemma Nuevo 1


In [15]:
pred_ensemble = (
    p_ens >= best_thresh
).astype(int)

gemma_falla_ensemble_acierta = (
    (pred_gemma != y_val) &
    (pred_ensemble == y_val)
)

print(
    "Ensemble rescata de Gemma:",
    gemma_falla_ensemble_acierta.sum()
)

gemma_1_falla_ensemble_acierta = (
    (pred_gemma_1 != y_val) &
    (pred_ensemble == y_val)
)

print(
    "Ensemble rescata de Gemma Nuevo:",
    gemma_falla_ensemble_acierta.sum()
)

qwen_falla_ensemble_acierta = (
    (pred_qwen != y_val) &
    (pred_ensemble == y_val)
)

print(
    "Ensemble rescata de Qwen:",
    qwen_falla_ensemble_acierta.sum()
)

roberta_falla_ensemble_acierta = (
    (pred_roberta != y_val) &
    (pred_ensemble == y_val)
)

print(
    "Ensemble rescata de RoBERTuito:",
    roberta_falla_ensemble_acierta.sum()
)

ensemble_mejora_gemma = (
    (pred_gemma != y_val) &
    (pred_ensemble == y_val)
)

ensemble_empeora_gemma = (
    (pred_gemma == y_val) &
    (pred_ensemble != y_val)
)

print(
    "Ensemble corrige errores de Gemma:",
    ensemble_mejora_gemma.sum()
)

print(
    "Ensemble introduce errores:",
    ensemble_empeora_gemma.sum()
)

Ensemble rescata de Gemma: 37
Ensemble rescata de Gemma Nuevo: 37
Ensemble rescata de Qwen: 35
Ensemble rescata de RoBERTuito: 55
Ensemble corrige errores de Gemma: 37
Ensemble introduce errores: 56


In [16]:
from sklearn.metrics import confusion_matrix, classification_report

print("Gemma")
print(classification_report(y_val, pred_gemma_1))
print(confusion_matrix(y_val, pred_gemma_1))

Gemma
              precision    recall  f1-score   support

           0       0.90      0.94      0.92       659
           1       0.88      0.81      0.85       381

    accuracy                           0.89      1040
   macro avg       0.89      0.87      0.88      1040
weighted avg       0.89      0.89      0.89      1040

[[618  41]
 [ 72 309]]


In [17]:
print("Ensamble")
print(classification_report(y_val, pred_ensemble))
print(confusion_matrix(y_val, pred_ensemble))

Ensamble
              precision    recall  f1-score   support

           0       0.93      0.86      0.89       659
           1       0.78      0.88      0.83       381

    accuracy                           0.87      1040
   macro avg       0.86      0.87      0.86      1040
weighted avg       0.88      0.87      0.87      1040

[[566  93]
 [ 44 337]]


Busqueda de 5 modelos (Descartado por decaimento de F1)

In [ ]:
import numpy as np
from sklearn.metrics import f1_score


p_gemma_old = probs_Gemma       
p_gemma_new = probs_Gemma_1     
p_roberta   = probs_betito      
p_Llama = probs_Osth        
p_llama = probs_Llama 


best_weights = (0.20, 0.20, 0.20, 0.20, 0.20)
best_thresh = 0.5
best_f1 = 0.0

print("Buscando la combinación óptima de pesos (5 modelos) y umbral...")


for w_gemma_old in np.arange(0.0, 1.01, 0.05):
    for w_gemma_new in np.arange(0.0, 1.01 - w_gemma_old, 0.05):
        
        suma_parcial_1 = w_gemma_old + w_gemma_new
        
        for w_roberta in np.arange(0.0, 1.01 - suma_parcial_1, 0.05):
            
            suma_parcial_2 = suma_parcial_1 + w_roberta
            
            for w_qwen_osth in np.arange(0.0, 1.01 - suma_parcial_2, 0.05):
                
                w_beto = 1.0 - w_gemma_old - w_gemma_new - w_roberta - w_qwen_osth
                
                if w_beto < -1e-9:
                    continue
                if w_beto < 0.0:
                    w_beto = 0.0  

                p_ens = (
                    (w_gemma_old * p_gemma_old) + 
                    (w_gemma_new * p_gemma_new) + 
                    (w_roberta * p_roberta) + 
                    (w_qwen_osth * p_Llama) +
                    (w_beto * p_llama)
                )

                # Búsqueda del umbral óptimo de clasificación
                for thresh in np.arange(0.3, 0.71, 0.02):
                    preds = (p_ens >= thresh).astype(int)

                    f1 = f1_score(y_val, preds, average='macro')
                    # Si encontramos un nuevo récord de F1, guardamos la alineación
                    if f1 > best_f1:
                        best_f1 = f1
                        best_weights = (w_gemma_old, w_gemma_new, w_roberta, w_qwen_osth, w_beto)
                        best_thresh = thresh

w_old, w_new, w_rob, w_qwen, w_bt = best_weights
print("\n¡Búsqueda de 5 vías completada con éxito! 🏆")
print("─" * 55)
print(f"Pesos óptimos encontrados:")
print(f"  • Gemma Anterior (p_gemma):    {w_old:.2f}")
print(f"  • Gemma Nuevo (p_gemma_1):    {w_new:.2f}")
print(f"  • RoBERTuito (p_roberta):      {w_rob:.2f}")
print(f"  • Qwen OSTH (p_qwen_osth):     {w_qwen:.2f}")
print(f"  • Llama (p_llama):               {w_bt:.2f}")
print(f"  (Suma total de pesos: {w_old + w_new + w_rob + w_qwen + w_bt:.2f})")
print("─" * 55)
print(f"Umbral de decisión (Threshold):  {best_thresh:.2f}")
print(f"Mejor Macro F1-Score en Val:     {best_f1:.4f}")
print("─" * 55)

Buscando la combinación óptima de pesos (5 modelos) y umbral...

¡Búsqueda de 5 vías completada con éxito! 🏆
───────────────────────────────────────────────────────
Pesos óptimos encontrados:
  • Gemma Anterior (p_gemma):    0.40
  • Gemma Nuevo (p_gemma_1):    0.10
  • RoBERTuito (p_roberta):      0.10
  • Qwen OSTH (p_qwen_osth):     0.00
  • Llama (p_llama):               0.40
  (Suma total de pesos: 1.00)
───────────────────────────────────────────────────────
Umbral de decisión (Threshold):  0.68
Mejor Macro F1-Score en Val:     0.8965
───────────────────────────────────────────────────────


In [ ]:
p_gemma_test = test_Gemma
p_gemma_test_1 = test_Gemma_1
p_roberta_test = test_betito
p_Llama = test_Osth
p_llama = test_Llama


p_ens_test = (best_weights[0] * p_gemma_test) + (best_weights[1] * p_gemma_test_1) + (best_weights[2] * p_roberta_test) + (best_weights[3] * p_Llama) + (best_weights[4] * p_llama)
preds_finales_test = (p_ens_test >= best_thresh).astype(int)

unique, counts = np.unique(preds_finales_test, return_counts=True)
print(dict(zip(unique.tolist(), counts.tolist())))

# Guardar CSV

NAME = 'GG_GG_Robertuito_WW_LL_V5.csv'

pd.DataFrame({
    'id'    : range(1, len(preds_finales_test) + 1),
    'klass' : preds_finales_test,
    #'score' : p_ensemble_test,
}).to_csv(NAME, index=False)

print(f'✅ {NAME} listo')

{0: 3690, 1: 1910}
✅ GG_GG_Robertuito_WW_LL_V5.csv listo


In [ ]:
print(f"Pesos óptimos: {best_weights}")
print(f"Umbral: {best_thresh:.3f}")

from scipy.stats import pearsonr
modelos = {
    'Gemma':      p_gemma_test,
    'Gemma_1':    p_gemma_test_1,
    'RoBERTuito': p_roberta_test,
    'Qwen_OSTH':  p_Llama,
}
import pandas as pd
scores_df = pd.DataFrame(modelos)
print("\nCorrelación entre modelos:")
print(scores_df.corr().round(3))

In [ ]:
import cma

p_gemma_val = probs_Gemma
p_gemma_val_1 = probs_Gemma_1
p_roberta_val = probs_betito
p_qwen_val = probs_Osth
val_labels = y_val
def objective(params):
    w  = np.array(params[:4])
    w  = np.abs(w) / np.abs(w).sum()  
    th = float(np.clip(params[4], 0.3, 0.8))
    
    p_ens = (w[0]*p_gemma_val + w[1]*p_gemma_val_1 +
             w[2]*p_roberta_val + w[3]*p_qwen_val)
    preds = (p_ens >= th).astype(int)
    return 1.0 - f1_score(val_labels, preds, average='macro')

x0     = list(best_weights) + [best_thresh]
sigma0 = 0.15 

es = cma.CMAEvolutionStrategy(x0, sigma0, {
    'maxiter'  : 500,
    'tolx'     : 1e-6,
    'tolfun'   : 1e-6,
    'seed'     : 42,
    'verbose'  : 1,
})
es.optimize(objective)

w_opt = np.abs(es.result.xbest[:4])
w_opt = w_opt / w_opt.sum()
th_opt = float(np.clip(es.result.xbest[4], 0.3, 0.8))
print(f"Pesos CMA-ES: {w_opt}")
print(f"Umbral CMA-ES: {th_opt:.4f}")

(4_w,8)-aCMA-ES (mu_w=2.6,w_1=52%) in dimension 5 (seed=42, Thu Jun  4 16:29:16 2026)
Iterat #Fevals   function value  axis ratio  sigma  min&max std  t[m:s]
    1      8 1.150923958534544e-01 1.0e+00 1.23e-01  1e-01  1e-01 0:00.0
    2     16 1.142921082498122e-01 1.3e+00 1.05e-01  9e-02  1e-01 0:00.0
    3     24 1.150923958534544e-01 1.4e+00 9.09e-02  7e-02  9e-02 0:00.0
   29    232 1.094101769499208e-01 4.5e+00 9.27e-03  3e-03  7e-03 0:00.3
termination on {'tolflatfitness': 1}
final/bestever f-value = 1.094102e-01 1.094102e-01 after 232/102 evaluations
incumbent solution: [0.03873719, 0.41808499, 0.08245487, 0.28183148, 0.63396399]
std deviations: [0.00727181, 0.00685328, 0.00305348, 0.00726742, 0.00255651]
Pesos CMA-ES: [0.0365175  0.51905866 0.09246359 0.35196026]
Umbral CMA-ES: 0.6295


In [40]:
import numpy as np

X_meta = np.column_stack([
    probs_beto,      # BETO
    probs_betito,    # RoBERTuito
    probs_Gemma,     # Gemma viejo
    probs_Gemma_1,   # Gemma nuevo
    probs_Osth       # Qwen OSTH
])

print(X_meta.shape)

(1040, 5)


In [37]:
from sklearn.metrics import f1_score

p_uniform = np.mean(X_meta, axis=1)

best_f1 = 0
best_thresh = 0

for thresh in np.arange(0.1, 0.91, 0.01):

    preds = (p_uniform >= thresh).astype(int)

    f1 = f1_score(
        y_val,
        preds,
        average='macro'
    )

    if f1 > best_f1:
        best_f1 = f1
        best_thresh = thresh

print("\nSOFT VOTING UNIFORME")
print("Threshold:", best_thresh)
print("F1:", best_f1)


SOFT VOTING UNIFORME
Threshold: 0.6099999999999998
F1: 0.8833632401948218


In [38]:
from sklearn.linear_model import LogisticRegression

meta_lr = LogisticRegression(
    max_iter=5000,
    class_weight="balanced"
)

meta_lr.fit(X_meta, y_val)

probs_lr = meta_lr.predict_proba(X_meta)[:,1]

best_f1_lr = 0
best_thresh_lr = 0

for thresh in np.arange(0.1,0.91,0.01):

    preds = (probs_lr >= thresh).astype(int)

    f1 = f1_score(
        y_val,
        preds,
        average='macro'
    )

    if f1 > best_f1_lr:
        best_f1_lr = f1
        best_thresh_lr = thresh

best_thresh = best_thresh_lr

print("\nLOGISTIC STACKING")
print("Threshold:", best_thresh)
print("F1:", best_f1_lr)


LOGISTIC STACKING
Threshold: 0.7299999999999996
F1: 0.8854707711021822


In [ ]:
model_names = [
    "BETO",
    "RoBERTuito",
    "Gemma_old",
    "Gemma_new",
    "Qwen_OSTH"
]

pesos = []

for name, coef in zip(
    model_names,
    meta_lr.coef_[0]
):
    print(name, coef)
    pesos.append(coef)

BETO 0.6809658567197644
RoBERTuito 0.8724116359482312
Gemma_old 1.1630486880281385
Gemma_new 1.2410563654296807
Qwen_OSTH 2.3775241632093858


In [ ]:
p_gemma_test = test_Gemma
p_gemma_test_1 = test_Gemma_1
p_roberta_test = test_betito
p_Llama = test_Osth
p_llama = beto_test


X_meta_test = np.column_stack((
    p_llama,
    p_roberta_test,
    p_gemma_test,
    p_gemma_test_1,
    p_Llama
))


probs_lr_test = meta_lr.predict_proba(X_meta_test)[:, 1]


preds_finales_test = (probs_lr_test >= best_thresh).astype(int)

NAME = 'GG_GG_Robertuito_WW_BETO_V2.csv'

pd.DataFrame({
    'id'    : range(1, len(preds_finales_test) + 1),
    'klass' : preds_finales_test,
    #'score' : p_ensemble_test,
}).to_csv(NAME, index=False)

print(f'{NAME} listo')


In [39]:
from sklearn.ensemble import RandomForestClassifier

meta_rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=5,
    random_state=42,
    n_jobs=-1
)

meta_rf.fit(X_meta, y_val)

probs_rf = meta_rf.predict_proba(X_meta)[:,1]
best_f1_rf = 0
best_thresh_rf = 0

for thresh in np.arange(0.1,0.91,0.01):

    preds = (probs_rf >= thresh).astype(int)

    f1 = f1_score(
        y_val,
        preds,
        average='macro'
    )

    if f1 > best_f1_rf:
        best_f1_rf = f1
        best_thresh_rf = thresh

print("\nRANDOM FOREST STACKING")
print("Threshold:", best_thresh_rf)
print("F1:", best_f1_rf)


RANDOM FOREST STACKING
Threshold: 0.4299999999999998
F1: 0.9183279902582072


In [34]:
model_names = [
    "Gemma Anterior (p_gemma)",
    "Gemma Nuevo (p_gemma_1)",
    "RoBERTuito (p_roberta)",
    "Qwen OSTH (p_qwen_osth)"
]

importancias = meta_rf.feature_importances_

print("── IMPORTANCIA DE CARACTERÍSTICAS (RANDOM FOREST) ──")
for name, imp in zip(model_names, importancias):
    print(f"  • {name}: {imp:.4f} ({imp*100:.1f}%)")
print("───────────────────────────────────────────────────")

── IMPORTANCIA DE CARACTERÍSTICAS (RANDOM FOREST) ──
  • Gemma Anterior (p_gemma): 0.0481 (4.8%)
  • Gemma Nuevo (p_gemma_1): 0.1453 (14.5%)
  • RoBERTuito (p_roberta): 0.2719 (27.2%)
  • Qwen OSTH (p_qwen_osth): 0.2722 (27.2%)
───────────────────────────────────────────────────


In [ ]:

X_meta_test = np.column_stack((
    p_llama,
    p_roberta_test,
    p_gemma_test,
    p_gemma_test_1,
    p_Llama
))

probs_rf_test = meta_rf.predict_proba(X_meta_test)[:, 1]

preds_finales_rf = (probs_rf_test >= best_thresh_rf).astype(int)

NAME = 'GG_GG_Robertuito_WW_BETO_RF_V1.csv'

pd.DataFrame({
    'id'    : range(1, len(preds_finales_test) + 1),
    'klass' : preds_finales_test,
    #'score' : p_ensemble_test,
}).to_csv(NAME, index=False)

print(f'{NAME} listo')


In [42]:
from xgboost import XGBClassifier

meta_xgb = XGBClassifier(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

meta_xgb.fit(X_meta, y_val)

probs_xgb = meta_xgb.predict_proba(X_meta)[:,1]

best_f1_xgb = 0
best_thresh_xgb = 0

for thresh in np.arange(0.1,0.91,0.01):

    preds = (probs_xgb >= thresh).astype(int)

    f1 = f1_score(
        y_val,
        preds,
        average='macro'
    )

    if f1 > best_f1_xgb:
        best_f1_xgb = f1
        best_thresh_xgb = thresh

print("\nXGBOOST STACKING")
print("Threshold:", best_thresh_xgb)
print("F1:", best_f1_xgb)


XGBOOST STACKING
Threshold: 0.44999999999999984
F1: 0.9216417170615644


In [ ]:
X_meta_test = np.column_stack((
    p_llama,
    p_roberta_test,
    p_gemma_test,
    p_gemma_test_1,
    p_Llama
))

probs_xgb_test = meta_xgb.predict_proba(X_meta_test)[:, 1]

preds_finales_xgb = (probs_xgb_test >= best_thresh_xgb).astype(int)

NAME = 'GG_GG_Robertuito_WW_BETO_XGB_V1.csv'

pd.DataFrame({
    'id'    : range(1, len(preds_finales_xgb) + 1),
    'klass' : preds_finales_xgb,
    #'score' : p_ensemble_test,
}).to_csv(NAME, index=False)

print(f'{NAME} listo')


In [ ]:
from sklearn.isotonic import IsotonicRegression

def calibrate(scores_train, labels_train, scores_test):
    ir = IsotonicRegression(out_of_bounds='clip')
    ir.fit(scores_train, labels_train)
    return ir.predict(scores_train), ir.predict(scores_test)


p_gemma_cal,   t_gemma_cal   = calibrate(probs_Gemma,    y_val, test_Gemma)
p_gemma1_cal,  t_gemma1_cal  = calibrate(probs_Gemma_1,  y_val, test_Gemma_1)
p_rob_cal,     t_rob_cal     = calibrate(probs_betito,    y_val, test_betito)
p_osth_cal,    t_osth_cal    = calibrate(probs_Osth,      y_val, test_Osth)
p_bert_cal,    t_bert_cal    = calibrate(probs_beto,      y_val, beto_test)

In [ ]:
import cma

scores_val_5 = np.column_stack([
    p_gemma_cal, p_gemma1_cal, p_rob_cal, p_osth_cal, p_bert_cal
])

def obj_5(params):
    w  = np.abs(params[:5]); w /= w.sum()
    th = float(np.clip(params[5], 0.30, 0.80))
    return 1.0 - f1_score(y_val, (scores_val_5 @ w >= th).astype(int), average='macro')


x0 = [0.037, 0.519, 0.092, 0.352, 0.10, 0.630]
es5 = cma.CMAEvolutionStrategy(x0, 0.12, {'maxiter': 800, 'seed': 42})
es5.optimize(obj_5)

w5 = np.abs(es5.result.xbest[:5]); w5 /= w5.sum()
th5 = float(np.clip(es5.result.xbest[5], 0.30, 0.80))
print(f'Pesos: {w5.round(4)} | Umbral: {th5:.4f}')
print(f'F1 val: {1 - es5.result.fbest:.4f}')



(4_w,8)-aCMA-ES (mu_w=2.6,w_1=52%) in dimension 5 (seed=42, Sat Jun  6 18:04:39 2026)
Iterat #Fevals   function value  axis ratio  sigma  min&max std  t[m:s]
    1      8 1.142921082498122e-01 1.0e+00 1.12e-01  1e-01  1e-01 0:00.0
    2     16 1.144096430984898e-01 1.3e+00 1.35e-01  1e-01  1e-01 0:00.0
    3     24 1.078592022567464e-01 1.7e+00 1.53e-01  1e-01  2e-01 0:00.0
   56    448 1.021723784145658e-01 1.5e+01 1.10e-02  1e-03  1e-02 0:00.7
termination on {'tolflatfitness': 1}
final/bestever f-value = 1.021724e-01 1.021724e-01 after 448/290 evaluations
incumbent solution: [0.04474248, 0.32290908, 0.18668836, 0.19068713, 0.52077757]
std deviations: [0.00900626, 0.01111469, 0.00351766, 0.00504178, 0.00096226]
Pesos: [0.0646 0.2506 0.1489 0.1719 0.3641] | Umbral: 0.5215
F1 val: 0.8978


In [ ]:

scores_test_5 = np.column_stack([test_Gemma, test_Gemma_1, test_betito, test_Osth, beto_test])
preds_test_5 = (scores_test_5 @ w5 >= th5).astype(int)

NAME = 'GG_GG_Robertuito_WW_CAL_CMA_V1.csv'

pd.DataFrame({
    'id'    : range(1, len(preds_test_5) + 1),
    'klass' : preds_test_5,
    #'score' : p_ensemble_test,
}).to_csv(NAME, index=False)

print(f'{NAME} listo')

In [ ]:
pred_g  = (probs_Gemma    >= best_thresh).astype(int) 
pred_g1 = (probs_Gemma_1  >= best_thresh).astype(int)
pred_rob = (probs_betito  >= best_thresh).astype(int)
pred_qw  = (probs_Osth    >= best_thresh).astype(int)

todos_fallan = (
    (pred_g  != y_val) &
    (pred_g1 != y_val) &
    (pred_rob != y_val) &
    (pred_qw  != y_val)
)
print(f"Ejemplos donde los 4 modelos fallan: {todos_fallan.sum()}")
print(f"De esos, cuántos son Humor no detectado (FN):")
print(f"  {((todos_fallan) & (y_val == 1)).sum()} FN compartidos")
print(f"  {((todos_fallan) & (y_val == 0)).sum()} FP compartidos")